In [1]:
import torch

from deepproblog.dataset import DataLoader
from deepproblog.engines import ExactEngine
from deepproblog.evaluate import get_confusion_matrix
from deepproblog.examples.Coins.data.dataset import train_dataset, test_dataset
from deepproblog.model import Model
from deepproblog.network import Network
from deepproblog.train import train_model
from deepproblog.utils.standard_networks import smallnet
from deepproblog.utils.stop_condition import Threshold, StopOnPlateau

batch_size = 5
loader = DataLoader(train_dataset, batch_size)
lr = 1e-4
coin_network1 = smallnet(num_classes=2, pretrained=True)
coin_net1 = Network(coin_network1, "net1", batching=True)
coin_net1.optimizer = torch.optim.Adam(coin_network1.parameters(), lr=lr)
coin_network2 = smallnet(num_classes=2, pretrained=True)
coin_net2 = Network(coin_network2, "net2", batching=True)
coin_net2.optimizer = torch.optim.Adam(coin_network2.parameters(), lr=lr)

model = Model("model.pl", [coin_net1, coin_net2])
model.add_tensor_source("train", train_dataset)
model.add_tensor_source("test", test_dataset)
model.set_engine(ExactEngine(model), cache=True)
train_obj = train_model(
    model,
    loader,
    StopOnPlateau("Accuracy", warm_up=10, patience=10)
    | Threshold("Accuracy", 1.0, duration=2),
    log_iter=100 // batch_size,
    test_iter=100 // batch_size,
    test=lambda x: [("Accuracy", get_confusion_matrix(x, test_dataset).accuracy())],
    infoloss=0.25,
)


/Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/engines/__init__.py:6: UserWarning: ApproximateEngine is not available as PySwip could not be found
  warnings.warn("ApproximateEngine is not available as PySwip could not be found")


Caching ACs


FileNotFoundError: [Errno 2] No such file or directory: '/Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/image_data/test/0.png'

In [3]:
import os
# 请将以下代码临时添加到您的脚本中验证路径
base_path = "/Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data"

# 检查目录结构
required_dirs = [
    f"{base_path}/image_data/train",
    f"{base_path}/image_data/test",
    f"{base_path}/label_data"
]

for d in required_dirs:
    if not os.path.exists(d):
        print(f"❌ 缺失关键目录: {d}")
    else:
        print(f"✅ 目录存在: {d}")

# 检查示例文件
sample_files = [
    f"{base_path}/image_data/test/0.png",
    f"{base_path}/label_data/test.csv"
]

for f in sample_files:
    if not os.path.isfile(f):
        print(f"⛔ 文件不存在: {f}")
    else:
        print(f"✔️ 文件存在: {f}")

❌ 缺失关键目录: /Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/image_data/train
❌ 缺失关键目录: /Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/image_data/test
✅ 目录存在: /Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/label_data
⛔ 文件不存在: /Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/image_data/test/0.png
✔️ 文件存在: /Users/zhenzhili/MASTERTHESIS/deepproblog/src/deepproblog/examples/Coins/data/label_data/test.csv
